# Train the BreachTrace report model (Colab)

Fine-tunes the specialised report writer and exports it as a 4-bit GGUF for the app.

1. **Runtime → Change runtime type → T4 GPU** (free tier is enough).
2. Run the cells top to bottom. Cell 2 asks you to upload two files from your repo:
   - `backend/research/report_train.py`
   - `data/report_sft.jsonl` (built by `python -m research.report_distill teach`)
3. The last cell downloads `breachtrace-reporter-q4_k_m.gguf` (~400 MB).

Then, on your machine, run the quality gate (`research/report_eval.py`, see README) before shipping it.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "torch" "transformers>=4.45" "peft>=0.12" "trl>=0.12" datasets accelerate

In [ ]:
from google.colab import files
uploaded = files.upload()   # select report_train.py and report_sft.jsonl
assert {"report_train.py", "report_sft.jsonl"} <= set(uploaded), sorted(uploaded)
!wc -l report_sft.jsonl

In [ ]:
# ~1 hour on a T4 for ~2.7k examples. Base model: Qwen2.5-0.5B-Instruct (Apache-2.0).
# If the gate later fails, re-run with --base Qwen/Qwen3-0.6B or Qwen/Qwen2.5-1.5B-Instruct.
!python report_train.py --data report_sft.jsonl --base Qwen/Qwen2.5-0.5B-Instruct --out reporter --epochs 3

In [ ]:
# Convert the merged model to GGUF and quantize to Q4_K_M.
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!python llama.cpp/convert_hf_to_gguf.py reporter/merged --outfile reporter-f16.gguf --outtype f16
!llama.cpp/build/bin/llama-quantize reporter-f16.gguf breachtrace-reporter-q4_k_m.gguf Q4_K_M
!ls -lh breachtrace-reporter-q4_k_m.gguf && sha256sum breachtrace-reporter-q4_k_m.gguf

In [ ]:
files.download("breachtrace-reporter-q4_k_m.gguf")